## Handling Multiple Tool Responses and Generating a Final User-Friendly 

In [83]:
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.tools import tool
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = ChatGroq(
    model='llama-3.1-8b-instant',
    temperature=0,
)

In [84]:
user_query = """
I want to go Cox's Bazar.
Please suggest me budget friendly flight,
recommend the best hotel,
tell me about weather
and estimate total cost.
"""

In [85]:
planner_prompt = f"""
You are a planning agent.

Available Tools:

1. weather_tool
   - weather information

2. flight_tool
   - flight information

3. hotel_tool
   - hotel recommendation

Your task:

- Extract destination
- Select required tools

Do not answer the user's question.

Return ONLY valid JSON.

Format:

{{
    "destination": "...",
    "required_tools": []
}}

User Query:

{user_query}
"""

planner_response = model.invoke(planner_prompt)

print(planner_response.content)

{
    "destination": "Cox's Bazar",
    "required_tools": ["flight_tool", "hotel_tool", "weather_tool"]
}


In [95]:
@tool
def weather_tool(destination: str) -> str:
    """
    Get weather information for a travel destination.
    """

    weather_data = [
        {"city": "Cox's Bazar", "weather": "Rainy", "temp": "28°C"},
        {"city": "Dhaka", "weather": "Cloudy", "temp": "31°C"},
        {"city": "Sylhet", "weather": "Thunderstorm", "temp": "26°C"},
        {"city": "Rangamati", "weather": "Sunny", "temp": "30°C"},
        {"city": "Bandarban", "weather": "Partly Cloudy", "temp": "27°C"},
    ]

    result = next(
        (
            item
            for item in weather_data
            if item["city"].lower() == destination.lower()
        ),
        None,
    )

    return result

In [100]:
@tool
def flight_tool(destination: str) -> str:
    """
    Get flight information for a destination.
    """

    flight_data = [
        {"destination": "Cox's Bazar", "price": "5500 BDT", "airline": "Safi Airline"},
        {"destination": "Cox's Bazar", "price": "6000 BDT", "airline": "Biman"},
        {"destination": "Sylhet", "price": "4200 BDT", "airline": "Biman"},
        {"destination": "Chattogram", "price": "3500 BDT", "airline": "NovoAir"},
        {"destination": "Jessore", "price": "3100 BDT", "airline": "US Bangla"},
        {"destination": "Saidpur", "price": "4800 BDT", "airline": "Biman"},
    ]

    matching_flights = [
        flight
        for flight in flight_data
        if flight["destination"].lower() == destination.lower()
    ]

    return matching_flights

In [102]:
@tool
def hotel_tool(destination: str) -> str:
    """
    Get hotel recommendations for a destination.
    """

    hotel_data = [
        {
            "city": "Cox's Bazar",
            "hotel": "Safi's Hotel",
            "rating": 4.7
        },
        {
            "city": "Cox's Bazar",
            "hotel": "Long Beach Hotel",
            "rating": 4.5
        },
        {
            "city": "Sylhet",
            "hotel": "Grand Sylhet",
            "rating": 4.8
        },
        {
            "city": "Bandarban",
            "hotel": "Hill View Resort",
            "rating": 4.4
        },
        {
            "city": "Rangamati",
            "hotel": "Parjatan Motel",
            "rating": 4.2
        }
    ]

    matching_hotels = [
        hotel
        for hotel in hotel_data
        if hotel["city"].lower() == destination.lower()
    ]

    return matching_hotels

In [103]:
tool_results = {
    "weather": weather_tool.invoke(
        {"destination": destination}
    ),
    "flight": flight_tool.invoke(
        {"destination": destination}
    ),
    "hotel": hotel_tool.invoke(
        {"destination": destination}
    ),
}

tool_results

{'weather': {'city': "Cox's Bazar", 'weather': 'Rainy', 'temp': '28°C'},
 'flight': [{'destination': "Cox's Bazar",
   'price': '5500 BDT',
   'airline': 'Safi Airline'},
  {'destination': "Cox's Bazar", 'price': '6000 BDT', 'airline': 'Biman'}],
 'hotel': [{'city': "Cox's Bazar", 'hotel': "Safi's Hotel", 'rating': 4.7},
  {'city': "Cox's Bazar", 'hotel': 'Long Beach Hotel', 'rating': 4.5}]}

In [104]:
final_prompt = f"""
User Query:

{user_query}

Tool Results:

{tool_results}

Create a final travel plan.
Recommend the best flight.
Recommend the best hotel.
Estimate total cost.

Return a user-friendly response.
"""

final_response = model.invoke(final_prompt)
print(final_response.content)

**Final Travel Plan to Cox's Bazar**

Based on the provided information, I've created a travel plan for you to Cox's Bazar.

**Best Budget-Friendly Flight:**
I recommend taking the **Safi Airline** flight, which costs **5500 BDT**. This is the most affordable option available.

**Best Hotel:**
I recommend staying at the **Safi's Hotel**, which has a rating of 4.7. It's a great option for a comfortable and enjoyable stay in Cox's Bazar.

**Weather:**
The weather in Cox's Bazar is currently **Rainy** with a temperature of **28°C**. Please pack accordingly and check the weather forecast before your trip.

**Estimated Total Cost:**
Based on the flight and hotel recommendations, the estimated total cost for your trip to Cox's Bazar is:

* Flight: 5500 BDT
* Hotel (per night): 2000 BDT (average price based on Safi's Hotel rating)
* Total: 7500 BDT (approximately $90 USD)

Please note that these estimates may vary depending on your specific travel dates and other factors.

**Travel Plan Summa